# Aula 4 : Funções


## Carregar os Arquivos

In [46]:
import datetime
import json
import os 

#Definir a pasta de trabalho para o diretório do script
# Pega o diretório de trabalho atual
pasta = os.getcwd()
print(pasta)

c:\Users\maria\OneDrive\Área de Trabalho\PROJETOSGITHUB\SOULCODE_PORTOTI_2026\AULAS\Aula 4


In [ ]:
def carregar_arquivo(nome_arquivo, valor_padrao, pasta_atual=pasta):
    """ 
    Tenta carregar um arquivo JSON. Se o arquivo não existir, retorna um valor padrão.

    """    
    caminho_completo = os.path.join(pasta_atual, nome_arquivo)
    try:
        with open(caminho_completo, "r", encoding="utf-8") as arquivo:
            return json.load(arquivo)
    except FileNotFoundError:
        return valor_padrao

In [48]:
def salvar_arquivo(nome_arquivo, dados):
    """ 
    Recebe um nome de arquivo e os dados e joga tudo para o HD
    
    """
    with open(nome_arquivo, "w", encoding="utf-8") as arquivo:
        json.dump(dados, arquivo, indent=4)

In [49]:
def exibir_menu_e_estoque (caixa_atual, estoque_atual):
    '''
    Função dedidada a exibir o menu e o estoque tela.

    '''

    print(f"""
===========================================================
                   LIVRARIA SOUL 
          Caixa Acumulado: R$ {caixa_atual:.2f}
============================================================""")
    
    print("ACERVO DISPONÍVEL: ")
    for id_livro, livro in enumerate(estoque_atual):
        print(f" [{id_livro}] - {livro['nome']} ({livro['ano']}) | Autor: {livro['autor']} - R$ {livro['preco']:.2f} | Estoque: {livro['quantidade']}")
    
    print("============MENU DE COMANDOS======================")
    print("--------------------------------------------------")
    print("'1' = Registrar Venda de Livro")
    print("'2' = Cadastrar Novo Livro")
    print("'3' = Sair do Sistema")
    print("'4' = Alterar Preço")
    print("'5' = Repor Estoque")
    print("'6' = Pesquisar por Nome/Autor")
    print("'7' = PROMOÇÕES")
    print("'8' = Nota Fiscal (Vendas da Sessão)")
    print("'9' = PAINEL DE ESTATÍSTICAS E BALANÇO")
    print("==================================================\n")


    
#Anotações para o futuro: O estoque pode existir e estar vazio
#Criar uma validação para avisar o usuário que o estoque está vazio e perguntar se quer prosseguir com isso mesmo







In [50]:
estoque_padrão = [
    {"nome": "O Alquimista", "autor": "Paulo Coelho", "ano": 1988, "preco": 45.00, "quantidade": 15},
    {"nome": "Dom Casmurro", "autor": "Machado de Assis", "ano": 1899, "preco": 35.00, "quantidade": 20},
]

In [51]:
estoque = carregar_arquivo("estoque.json", estoque_padrão)
historico_vendas = carregar_arquivo("historico_vendas.json", [])

caixa = sum(venda['preco'] for venda in historico_vendas)

In [ ]:
while True:
    exibir_menu_e_estoque(caixa, estoque)

    comando = int(input("Digite o número do comando desejado: "))

    if comando == 3: 
        print("Encerrando o sistema. Total geral em caixa: R$ {:.2f}".format(caixa))
        break

    elif comando == 1:
        id_venda = int(input("Digite o ID do livro vendido: ").strip())
        if 0 <= id_venda < len(estoque):
            print(("ERRO: ID INVALIDO"))
        else:
            qtd = int(input(f"Quantos exemplares de {estoque[id_venda]['nome']} foram vendidos? ").strip())
            if qtd <=0:
                print("ERRO: Quantidade deve ser maior que zero.")
            elif qtd > estoque[id_venda]['quantidade']:
                print(f"ERRO: Estoque insuficiente. Apenas {estoque[id_venda]['quantidade']} disponíveis.")
            else:
                estoque[id_venda]['quantidade'] -= qtd
                valor_total = qtd * estoque[id_venda]['preco'] 

                caixa += valor_total

                # %d%,%m/%Y 
                # %H:%M:%S

                historico_vendas.append({
                    "item": estoque[id_venda]['nome'],
                    "autor": estoque[id_venda]['autor'],
                    "valor": valor_total,
                    "quantidade": qtd,
                    "horarios": datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")
                })

    elif comando == 2:
        print("\n========== CADASTRAR NOVO LIVRO ==========")
        novo_nome = input("Qual o nome do livro? ").strip()
        novo_autor = input("Quem é o autor do livro? ").strip()
        novo_ano = int(input("Qual o ano de lançamento do livro? ").strip())    
        novo_preco = float(input("Qual o preço de venda do livro? ").strip())    
        nova_quantidade = int(input("Qual a quantidade em estoque? ").strip())

        estoque.append({
            "nome": novo_nome,
            "autor": novo_autor,
            "ano": novo_ano,
            "preco": novo_preco,
            "quantidade": nova_quantidade
        })

        salvar_arquivo("estoque.json", estoque)
        print(f"Livro '{novo_nome}' cadastrado com sucesso!")

    
    elif comando == 4:
        id_produto = int(input("Digite o [ID] do livro para alterar o preço: "))
        if id_produto < 0 or id_produto >= len(estoque):
            print("ERRO: ID inválido.")
        else:
            novo_preco = float(input(f"Digite o novo preço para '{estoque[id_produto]['nome']}': "))
            if novo_preco <= 0:
                print("ERRO: O preço deve ser maior que zero.")
            else:
                estoque[id_produto]['preco'] = novo_preco
                salvar_arquivo("estoque.json", estoque)
                print(f"Preço de '{estoque[id_produto]['nome']}' atualizado para R$ {novo_preco:.2f} com sucesso!")
        
    elif comando == 5:
        id_produto = int(input("Digite o [ID] do livro para repor o estoque: "))
        if id_produto < 0 or id_produto >= len(estoque):
            print("ERRO: ID inválido.")
        else:
            quantidade_repor = int(input(f"Digite a quantidade a ser adicionada ao estoque de '{estoque[id_produto]['nome']}': "))
            if quantidade_repor <= 0:
                print("ERRO: A quantidade a repor deve ser maior que zero.")
            else:
                estoque[id_produto]['quantidade'] += quantidade_repor
                salvar_arquivo("estoque.json", estoque)
                print(f"Estoque de '{estoque[id_produto]['nome']}' atualizado. Quantidade atual: {estoque[id_produto]['quantidade']} unidades.")

    
    elif comando == 6:
        print("\n========== PESQUISAR POR NOME/AUTOR ==========")

        termo_pesquisa = input("Digite o nome ou autor para pesquisar: ").strip().lower()
        resultado = False

        for livro in estoque:
            if termo_pesquisa in livro['nome'].lower() or termo_pesquisa in livro['autor'].lower() or termo_pesquisa in str(livro['ano']):
                print(f" - {livro['nome']} ({livro['ano']}) | Autor: {livro['autor']} - R$ {livro['preco']:.2f} | Estoque: {livro['quantidade']}")
                resultado = True
        if not resultado:
            print("Nenhum livro encontrado com esse termo de pesquisa.")

    elif comando == 7:
        print("\n========== PROMOÇÕES ==========")
        print("  1 - Desconto em 1 produto específico")
        print("  2 - Desconto em todo o cardápio")

        tipo_promocao = int(input("Digite o número do tipo de promoção desejada: "))

        porcentagem_desconto = float(input("Digite a porcentagem de desconto (ex: 10 para 10%): "))

        fator_desconto = 1 - (porcentagem_desconto / 100)

        if tipo_promocao == 1:
            id_produto = int(input("Digite o [ID] do livro para aplicar o desconto: "))
            if id_produto < 0 or id_produto >= len(estoque):
                print("ERRO: ID inválido.")
            else:
                estoque[id_produto]['preco'] = round(estoque[id_produto]['preco'] * fator_desconto, 2)
                salvar_arquivo("estoque.json", estoque)
                print(f"Desconto aplicado! Novo preço de '{estoque[id_produto]['nome']}': R$ {estoque[id_produto]['preco']:.2f}")

        elif tipo_promocao == 2:
            for livro in estoque:
                livro['preco'] = round(livro['preco'] * fator_desconto, 2)
            salvar_arquivo("estoque.json", estoque)
            print(f"Desconto de {porcentagem_desconto:.2f}% aplicado a todos os livros! Confira os novos preços no menu principal.")
        
        else:
            print("ERRO: Tipo de promoção inválido.")
        
    elif comando == 8:
        print("\n========== NOTA FISCAL ==========")
        if not historico_vendas:
            print("Nenhuma venda registrada ainda.")
        else:
            for i, venda in enumerate(historico_vendas):
                print(f"[{i}] Data/Hora: {venda['horarios']} - {venda['quantidade']} x {venda['item']} | Total venda:  R$ {venda['valor']:.2f} ")
            

        


                   LIVRARIA SOUL 
          Caixa Acumulado: R$ 0.00
ACERVO DISPONÍVEL: 
 [0] - Teste (2026) | Autor: Teste - R$ 30.00 | Estoque: 100
 [1] - Teste (2020) | Autor: Teste - R$ 40.00 | Estoque: 100
============MENU DE COMANDOS======================
--------------------------------------------------
'1' = Registrar Venda de Livro
'2' = Cadastrar Novo Livro
'3' = Sair do Sistema
'4' = Alterar Preço
'5' = Repor Estoque
'6' = Pesquisar por Nome/Autor
'7' = PROMOÇÕES
'8' = Nota Fiscal (Vendas da Sessão)
'9' = PAINEL DE ESTATÍSTICAS E BALANÇO


========== CADASTRAR NOVO LIVRO ==========
Livro 'Crime e Castigo' cadastrado com sucesso!

                   LIVRARIA SOUL 
          Caixa Acumulado: R$ 0.00
ACERVO DISPONÍVEL: 
 [0] - Teste (2026) | Autor: Teste - R$ 30.00 | Estoque: 100
 [1] - Teste (2020) | Autor: Teste - R$ 40.00 | Estoque: 100
 [2] - Crime e Castigo (1968) | Autor: Friedor Dostoievsky - R$ 20.00 | Estoque: 500
============MENU DE COMANDOS======================
--